# Test your setup

This notebook checks that your environment is configured correctly — it imports the workshop helpers, loads your `.env`, and runs a tiny agent against your chosen model provider. If the cells below run cleanly, you're ready to start the workshop.

## 1. Pick the right kernel

Before running any cell, you need to point this notebook at the Python environment where your workshop dependencies are installed. Open the kernel picker (top-right of the notebook in VS Code, or **Kernel → Change kernel** in Jupyter) and choose:

| If you set up via... | Pick this kernel |
|---|---|
| **Codespace / Dev Container** | **Workshop (Python 3.12)** (under *Jupyter Kernels*, **not** *Python Environments*) |
| **Local virtual environment** | The Python interpreter inside your `.venv` (e.g. `Python 3.x ('.venv': venv)`) |

<br>

> Don't see the kernel in the list? In VS Code, run **Python: Select Interpreter** from the Command Palette (`Cmd/Ctrl + Shift + P`) and pick `/opt/venv/bin/python` — the kernel picker will then offer it.

## 2. Install Requirements

Run each cell in order. If the import cell fails with `ImportError: cannot import name 'load_env' from 'utils'`, you're either on the wrong kernel or you skipped `pip install -e .` during local setup — go back and check both.

In [ ]:
# Load environment variables into journal
import os
from utils import load_env
load_env()

---
## 1. A simple agent

The simplest agent is just a name and a set of instructions. The instructions are the agent's personality and brief — they decide how it sounds and what it cares about.

Try changing the instructions below to see how the same question gets a very different answer.

In [ ]:
# Create a Chat Client handle the sending of your messages to the model
# Reference: https://learn.microsoft.com/en-us/agent-framework/agents/?pivots=programming-language-python#supported-chat-providers

if os.getenv("MODEL_PROVIDER") == 'foundry':
    from agent_framework.foundry import FoundryChatClient
    from azure.core.credentials import AccessToken
    from azure.core.credentials_async import AsyncTokenCredential

    api_key = os.getenv("FOUNDRY_API_KEY")

    class _ApiKeyCredential(AsyncTokenCredential):
        """Wraps an API key as a TokenCredential for Foundry."""
        async def get_token(self, *scopes, **kwargs):
            return AccessToken(api_key, 0) # type: ignore
        async def close(self): pass
        async def __aenter__(self): return self
        async def __aexit__(self, *args): pass

    credential = _ApiKeyCredential()

    client = FoundryChatClient(
        project_endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
        model=os.getenv("FOUNDRY_MODEL"),
        credential=credential,
    )
elif os.getenv("MODEL_PROVIDER") == 'ollama':
    from agent_framework.ollama import OllamaChatClient

    client = OllamaChatClient(model=os.getenv("OLLAMA_CHAT_MODEL_ID"))
else:
    raise ValueError(
        "MODEL_PROVIDER is not set or not recognised. "
        "Open your .env file and set MODEL_PROVIDER to 'foundry' or 'ollama', then re-run this cell."
    )


In [ ]:
# Use the chat client to create an Agent instance
# Reference: https://learn.microsoft.com/en-us/agent-framework/agents/?pivots=programming-language-python#simple-agents-based-on-inference-services-1
from agent_framework import Agent

agent = Agent(
    client=client,  # type: ignore
    name="Simple Southern Agent",
    instructions="You are my assistant. Answer my questions in a southern accent with southern charm.",
)

In [ ]:
# Ask the agent a question and stream the response to your terminal
async for chunk in agent.run("How is the weather in Perth this weekend?", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()


---
## 2. Giving your agent tools

An agent on its own only knows what's in its training data. Tools let it reach beyond that — call a function, look something up, take an action in the world.

First, see what happens when a news agent has no tools and is asked about today's headlines. Then give it a `get_todays_headlines` tool and watch the response change.

In [ ]:
# Create and agent with no tools to see what it is capable of
agent = Agent(
    client=client, # type: ignore
    name="News Agent",
    instructions="You are a helpful news assistant. Answer questions about current events.",
)

async for chunk in agent.run("What's in the news today?", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

In [ ]:
# Create a custom tool which the agent can use to get the news
def get_todays_headlines() -> str:
    return """
    1. Global reforestation effort plants one billion trees, smashing annual record
    2. New cancer vaccine shows 90 percent success rate in landmark clinical trial
    3. Remote Australian town powered entirely by renewable energy for first time
    4. Endangered whale population rebounds to highest count in 50 years
    5. Local community garden program credited with cutting food insecurity by half
    """

In [ ]:
# See what the agent is capable of when you give it tools
agent = Agent(
    client=client, # type: ignore
    name="News Agent",
    instructions="You are a helpful news assistant. Answer questions about current events.",
    tools=[get_todays_headlines]
)

async for chunk in agent.run("What's in the news today?", stream=True):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()